# Day 25：LangChain 初探 — 统一 ChatModel 与 ChatPromptTemplate

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第三阶段第一天（Day 25）！今天我们将基于自研的轻量级 `zhishu_agent.langchain_core`，实战体验 LangChain 的三大基石：`BaseMessage` 消息协议、`ChatPromptTemplate.from_messages(...)` 提示词编排，以及 `BaseChatModel` 统一多模型网关在 DeepSeek 与 Qwen 之间的无缝切换（`v0.3.0-day25`）。

## 1. 环境初始化与导入 `project_iterations/day_25` 模块
自动定位仓库根目录并将 `project_iterations/day_25` 加入 `sys.path`，实现零外部重依赖离线运行。

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day25_dir = repo_root / "project_iterations" / "day_25"
if str(day25_dir) not in sys.path:
    sys.path.insert(0, str(day25_dir))

from zhishu_agent import (
    AIMessage,
    BaseChatModel,
    ChatPromptTemplate,
    DeepSeekChatModel,
    HumanMessage,
    QwenChatModel,
    SystemMessage,
    get_chat_model,
)

print("Day 25 模块加载成功:", day25_dir)

Day 25 模块加载成功: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_25


## 2. 标准化消息对象：`SystemMessage` / `HumanMessage` / `AIMessage`
验证消息类的 `role`、`type`、人类可读调试格式 `pretty_repr()` 与标准 API 字典格式 `to_dict()`。

In [2]:
conversation = [
    SystemMessage(content="你是星辰数科企业级智能体架构师。"),
    HumanMessage(content="为什么不能在业务代码里硬编码各家大模型的 HTTP 请求？"),
    AIMessage(content="因为不同厂商的协议、字段和重试机制各异，必须通过抽象层统一收敛。"),
]

for msg in conversation:
    print(msg.pretty_repr())

print("\n标准 API 序列化载荷:")
print([msg.to_dict() for msg in conversation])

[SYSTEM] 你是星辰数科企业级智能体架构师。
[USER] 为什么不能在业务代码里硬编码各家大模型的 HTTP 请求？
[ASSISTANT] 因为不同厂商的协议、字段和重试机制各异，必须通过抽象层统一收敛。

标准 API 序列化载荷:
[{'role': 'system', 'content': '你是星辰数科企业级智能体架构师。'}, {'role': 'user', 'content': '为什么不能在业务代码里硬编码各家大模型的 HTTP 请求？'}, {'role': 'assistant', 'content': '因为不同厂商的协议、字段和重试机制各异，必须通过抽象层统一收敛。'}]


## 3. 声明式提示词编排：`ChatPromptTemplate.from_messages(...)`
支持 `("system", ...)`、`("human", ...)` 元组简写以及直接嵌入静态 `BaseMessage` 实例，并自动提取 `input_variables`。

In [3]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是星辰数科 {team} 的资深顾问，请以 {style} 风格作答。"),
        ("human", "请评估我们在核心链路引入 {technology} 的收益。"),
    ]
)

print("自动识别的模板变量:", prompt.input_variables)
prompt_value = prompt.invoke(
    {
        "team": "AI 平台组",
        "style": "简洁工程化",
        "technology": "LangChain ChatModel 抽象层",
    }
)
print("\n渲染后的消息文本:\n" + prompt_value.to_string())

自动识别的模板变量: ['team', 'style', 'technology']

渲染后的消息文本:
[SYSTEM] 你是星辰数科 AI 平台组 的资深顾问，请以 简洁工程化 风格作答。
[USER] 请评估我们在核心链路引入 LangChain ChatModel 抽象层 的收益。


## 4. 统一多模型网关：零代码侵入切换 DeepSeek 与 Qwen
使用同一份 `prompt_value`，分别通过 `DeepSeekChatModel`、`QwenChatModel` 与 `get_chat_model()` 工厂进行推理并对比 `response_metadata`。

In [4]:
def run_with_gateway(model: BaseChatModel) -> None:
    reply = model.invoke(prompt_value)
    print(f"=== 厂商: {model.provider} | 模型: {model.model_name} ===")
    print(reply.content)
    print("响应元数据:", reply.response_metadata)
    print()

for provider in ("deepseek", "qwen"):
    run_with_gateway(get_chat_model(provider))

# 演示批量推理 batch()
qwen_max = QwenChatModel(model_name="qwen-max")
batch_replies = qwen_max.batch(["如何做模型降级？", "如何统计 Token 成本？"])
print("批量推理结果条数:", len(batch_replies))

=== 厂商: deepseek | 模型: deepseek-chat ===
[DeepSeek/deepseek-chat]（遵循系统设定：你是星辰数科 AI 平台组 的资深顾问，请以 简洁工程化 风格作答。）
针对您的输入「请评估我们在核心链路引入 LangChain ChatModel 抽象层 的收益。」，已完成结构化推演：共接收 2 条上下文消息，建议采用分层解耦架构并保持接口契约稳定。
响应元数据: {'provider': 'deepseek', 'model_name': 'deepseek-chat', 'temperature': 0.2, 'message_count': 2, 'usage': {'prompt_tokens': 75, 'completion_tokens': 160}}

=== 厂商: qwen | 模型: qwen-plus ===
[Qwen/qwen-plus]（角色设定：你是星辰数科 AI 平台组 的资深顾问，请以 简洁工程化 风格作答。）
已收到问题「请评估我们在核心链路引入 LangChain ChatModel 抽象层 的收益。」（上下文共 2 轮次）。通义千问为您生成企业级稳健方案：通过统一网关屏蔽底层模型差异，实现零代码侵入切换。
响应元数据: {'provider': 'qwen', 'model_name': 'qwen-plus', 'temperature': 0.3, 'message_count': 2, 'usage': {'prompt_tokens': 75, 'completion_tokens': 159}}

批量推理结果条数: 2
